In [2]:
import math
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from representation import represent_heatmap
from representation import represent_heatmap_overlaid 
from representation import represent_isolines 
from representation import represent_isolines_superimposed 
from representation import represent_hard_selection 
from representation import represent_soft_selection


r_resolution = 1200 / 325 
d_distance = 900 
alpha_radians = math.radians(2) 
sigma = r_resolution * d_distance * math.tan(alpha_radians)

def generate_partial_saliency_map(width, height, x_fix, y_fix, sigma_val):
    x_coords = np.arange(0, width, 1) 
    y_coords = np.arange(0, height, 1) 
    x_grid, y_grid = np.meshgrid(x_coords, y_coords) 
    x_term = ((x_grid - x_fix) ** 2) / (2 * (sigma_val ** 2))
    y_term = ((y_grid - y_fix) ** 2) / (2 * (sigma_val ** 2))
    return np.exp(-(x_term + y_term))

def generate_global_saliency_map(width, height, fixations, sigma_val):
    global_map = np.zeros((height, width)) 
    for x_fix, y_fix in fixations:
        global_map += generate_partial_saliency_map(width, height, x_fix, y_fix, sigma_val)
    max_val = np.max(global_map)
    if max_val > 0:
        global_map = global_map / max_val
    return global_map

original_image = Image.open("test.jpg").convert("RGB")

saliency_img = Image.open("test_saliency_img.png").convert("L")
saliency_map = np.array(saliency_img).astype(np.float32) / 255.0

cmap = plt.get_cmap("jet")
THRESHOLD = 0.4

img_heat = represent_heatmap(saliency_map, cmap)
img_heat_overlaid = represent_heatmap_overlaid(saliency_map, original_image, cmap)
img_iso = represent_isolines(saliency_map, cmap)
img_iso_overlaid = represent_isolines_superimposed(saliency_map, original_image, cmap)
img_hard = represent_hard_selection(saliency_map, original_image, THRESHOLD)
img_soft = represent_soft_selection(saliency_map, original_image, THRESHOLD)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0, 0].imshow(img_heat)
axes[0, 0].set_title("Heatmap")

axes[0, 1].imshow(img_heat_overlaid)
axes[0, 1].set_title("Overlaid Heatmap")

axes[0, 2].imshow(img_iso)
axes[0, 2].set_title("Isolines")

axes[1, 0].imshow(img_iso_overlaid)
axes[1, 0].set_title("Superimposed Isolines")

axes[1, 1].imshow(img_hard)
axes[1, 1].set_title(f"Hard Selection (t={THRESHOLD})")

axes[1, 2].imshow(img_soft)
axes[1, 2].set_title("Soft Selection")

for ax in axes.flat:
    ax.axis('off')

plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'representation'